# Data Science u kulturi — praktične vježbe
### Colab uz kolegij | ak. god. 2026./2027.

**Nositelj:** izv. prof. dr. sc. Benedikt Perak

Ovaj Colab oprimjeruje cijeli data-science pipeline u kulturološkim istraživanjima:
prikupljanje → čišćenje → analiza → vizualizacija → interpretacija,
uz primjenu **AI alata (LLM)** gdje su relevantni.

**Povezano:** Perak, B. (2025). *Komunikacija u doba umjetne inteligencije*. FFRI. ([GitHub](https://github.com/bperak/komunikacija_u_doba_ai))

---

## Sadržaj
1. Uvod u Python i Google Colab
2. Tablični podaci s Pandas (CSV, Excel)
3. Statistička analiza (deskriptivna, korelacije)
4. Vizualizacija (Matplotlib, Seaborn)
5. NLP obrada teksta + LLM analiza
6. Vježbe 🟢🟡🏆

---
## 0. Postavljanje

Instaliramo biblioteke. **Ključ (za LLM dio):** [aistudio.google.com](https://aistudio.google.com) → API key (besplatno).

In [ ]:
# @title Instalacija i setup
!pip install -q pandas numpy matplotlib seaborn scipy scikit-learn google-generativeai

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Tema grafova
plt.style.use('dark_background')
sns.set_palette("husl")

import os
from google.colab import userdata
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    os.environ['GEMINI_API_KEY'] = input("Gemini API ključ: ")

print("✅ Spremno:", pd.__version__)

---
## 1. Pandas — tablični podaci

U kulturološkim istraživanjima podaci su često u tablicama (ankete, katalozi, metapodaci).
**DataFrame** je osnovna struktura: retci = opažanja, stupci = varijable.

In [ ]:
# @title Kreiramo DataFrame iz anketnih podataka
# Primjer: anketa o kulturnim navikama studenata
data = {
    "id": range(1, 11),
    "dob": [19, 21, 20, 22, 19, 23, 20, 21, 22, 20],
    "kino_godisnje": [12, 3, 8, 15, 5, 20, 10, 7, 2, 14],
    "citam_tjedno_sati": [5, 2, 8, 3, 1, 10, 4, 6, 2, 7],
    "glazbeni_koncerti": [4, 1, 2, 6, 0, 8, 3, 2, 1, 5],
}
df = pd.DataFrame(data)
df["kultura_index"] = df["kino_godisnje"] + df["glazbeni_koncerti"]

print(df)
print("\n--- Deskriptivna statistika ---")
print(df[["kino_godisnje", "citam_tjedno_sati", "kultura_index"]].describe().round(2))

---
## 2. Statistička analiza

**Korelacija** mjeri povezanost dviju varijabli (od -1 do +1).
Važno: korelacija ≠ uzročnost!

In [ ]:
# @title Korelacijska analiza
import scipy.stats as stats

print("=== Korelacije (Pearson) ===")
print(df[["dob", "kino_godisnje", "citam_tjedno_sati", "glazbeni_koncerti"]].corr().round(2))

# Test: povezanost kino i citanja
r, p = stats.pearsonr(df["kino_godisnje"], df["citam_tjedno_sati"])
print(f"\nkino vs citanje: r={r:.3f}, p={p:.3f}")
print("Zaključak:", "statistički značajno" if p < 0.05 else "nije statistički značajno")

---
## 3. Vizualizacija

"Grafikon vrijedi 1000 riječi" — vizualizacija je ključni dio znanstvenog izvješća.
Pravila: jasne oznake, čitljiv font, poštena skala.

In [ ]:
# @title Vizualizacija kulturnih navika
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram
axes[0].hist(df["kino_godisnje"], bins=6, color="#4fc3f7", edgecolor="white")
axes[0].set_title("Kino posjeti godišnje")
axes[0].set_xlabel("broj posjeta")

# Scatter s regresijom
axes[1].scatter(df["citam_tjedno_sati"], df["kino_godisnje"], color="#ffb74d", s=60)
axes[1].set_title("Čitanje vs kino")
axes[1].set_xlabel("čitanje (h/tjedan)")
axes[1].set_ylabel("kino (posjeta/god)"

# Bar chart
axes[2].bar(df["id"], df["kultura_index"], color="#81c784")
axes[2].set_title("Kultura index po studentu")
axes[2].set_xlabel("student id")

plt.tight_layout()
plt.show()

print("Vizualizacija spremna za istraživačko izvješće.")

---
## 4. NLP obrada teksta

Tekstualni podaci (novinski članci, društvene mreže, arhivi) čine velik dio
humanističkih podataka. Osnovni koraci: tokenizacija, uklanjanje stop-riječi, frekvencije.

In [ ]:
# @title Osnove NLP-a: frekvencija riječi
import re
from collections import Counter

tekst = """
Kultura se prenosi kroz jezik, a jezik oblikuje kulturu.
Umjetna inteligencija mijenja način na koji komuniciramo.
Jezik i kultura su neodvojivi, a digitalni alati otvaraju nova pitanja.
"""

# Čišćenje i tokenizacija
rijeci = re.findall(r"\b[a-zćčžšđ]+\b", tekst.lower())
stop = {"i", "a", "se", "na", "u", "koji", "kroz", "koja"}
rijeci_bez_stop = [r for r in rijeci if r not in stop]

freq = Counter(rijeci_bez_stop)
print("Top 10 riječi:")
for rijec, n in freq.most_common(10):
    print(f"  {rijec}: {n}")

# Vizualizacija
import matplotlib.pyplot as plt
top = freq.most_common(8)
plt.figure(figsize=(10, 4))
plt.bar([w for w,_ in top], [n for _,n in top], color="#ba68c8")
plt.title("Frekvencija riječi")
plt.show()

---
## 5. LLM analiza — AI alati u podatkovnoj znanosti

LLM-ovi ubrzavaju: označavanje podataka, sažimanje, klasifikaciju, generiranje opisa.
**Kritički pristup:** uvijek provjeriti rezultate — modeli mogu halucinirati.

In [ ]:
# @title LLM: klasifikacija sentimenta kulturnih sadržaja
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel("gemini-2.0-flash")

recenzije = [
    "Izložba je bila fantastična, preporučujem svima!",
    "Konceri su ove godine razočarali, zvuk je bio loš.",
    "Novi film je zanimljiv, ali predvidljiv.",
]

prompt = f"""
Klasificiraj sentiment svake recenzije kao pozitivan, negativan ili neutralan.
Vrati JSON: {{"recenzije": [{{"tekst": "...", "sentiment": "...", "vjerojatnost": 0.9}}]}}

Recenzije:
{recenzije}
"""

odgovor = model.generate_content(prompt)
print(odgovor.text)

---
## 6. Vježbe

### 🟢 Osnovna
1. Učitaj vlastiti CSV (npr. anketa) i izračunaj deskriptivnu statistiku.
2. Napravi 3 vizualizacije svojih podataka s pravilnim oznakama.

### 🟡 Srednja
3. Preuzmi podatke s API-ja (npr. Wikipedia, YouTube) i napravi analizu frekvencija.
4. Usporedi ručno označavanje s LLM označavanjem na 10 primjera — izračunaj podudarnost.

### 🏆 Napredna
5. Izgradi kompletan pipeline: prikupljanje (API) → Pandas → vizualizacija → LLM interpretacija → izvješće.
6. Spoji agenta s MCP alatom (Google ADK 2.6) koji automatski analizira kulturni sadržaj.

---
### Resursi
- Grus, J. (2019). *Data Science from Scratch*. O'Reilly.
- Python za lingviste: https://github.com/nljubesi/python-for-linguists
- Programming with Python for the Humanities: https://www.karsdorp.io/python-course/
- Perak, B. (2025). *Komunikacija u doba umjetne inteligencije*. FFRI. [GitHub](https://github.com/bperak/komunikacija_u_doba_ai)
- Google AI Studio: https://aistudio.google.com

*Kolegij: Data Science u kulturi | 2026./2027.*